# Observatoire immobilier — requêtes Gold

Ce notebook utilise uniquement les quatre tables du schéma conceptuel : `G_COMMUNE`, `G_TRANSACTION`, `G_POINT_INTERET` et `G_PROFIL_COMMUNE`. Les besoins livrés sont B2, B4 et B5. Les calculs sont dans les fichiers SQL du dossier `sql/gold/`.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from gold.requetes import charger_requete

load_dotenv(ROOT / '.env')
engine = create_engine(os.environ['DATABASE_URL'], pool_pre_ping=True)
REGIONS = {'idf': 'Île-de-France', 'ara': 'Auvergne-Rhône-Alpes',
           'bretagne': 'Bretagne', 'occitanie': 'Occitanie'}


In [ ]:
def executer(besoin, **parametres):
    with engine.connect() as connexion:
        return pd.read_sql(charger_requete(besoin), connexion, params=parametres)

with engine.connect() as connexion:
    from sqlalchemy import text
    tables = connexion.execute(text("SELECT tablename FROM pg_tables WHERE schemaname = current_schema() AND tablename LIKE 'G_%' ORDER BY tablename")).scalars().all()
print('Tables Gold :', tables)


## B2 — Implantation d’agences

Le score associe la dynamique des prix, l’activité récente et les densités de points d’intérêt de la même région. Les résultats sont calculés directement depuis les quatre tables Gold.


In [ ]:
classements = []
for code_region, nom_region in REGIONS.items():
    resultats = executer('B2', region=code_region, limite=10)
    resultats['nom_region'] = nom_region
    classements.append(resultats)
b2 = pd.concat(classements, ignore_index=True)
display(b2[['nom_region', 'rang_dans_la_region', 'nom_commune',
            'score_attractivite', 'nb_ventes_2024_2025', 'nb_poi_osm']])

fig, ax = plt.subplots(figsize=(10, 5))
b2.groupby('nom_region')['score_attractivite'].mean().astype(float).sort_values().plot.barh(ax=ax)
ax.set_xlabel('Score moyen des dix premières communes')
ax.set_ylabel('')
plt.tight_layout()
plt.show()


## B4 — Tendance du marché

Les prix annuels et leurs variations sont recalculés depuis `G_TRANSACTION`. Les prix restent absents sous dix ventes.


In [ ]:
b4 = executer('B4', code_commune='75056', type_bien='Appartement')
display(b4)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(b4['annee'], pd.to_numeric(b4['prix_m2_median']), marker='o')
ax.set_title('Paris — prix médian des appartements')
ax.set_xlabel('Année')
ax.set_ylabel('€/m²')
plt.tight_layout()
plt.show()


## B5 — Communes à prix bas et bien équipées

La comparaison utilise des communes de la même région et du même type de logement, avec au moins dix ventes récentes et un revenu médian renseigné.


In [ ]:
zones = []
for code_region, nom_region in REGIONS.items():
    resultats = executer('B5', region=code_region, type_bien='Maison',
                         ratio_prix_max=0.90, score_equipements_min=60, limite=10)
    resultats['nom_region'] = nom_region
    zones.append(resultats)
b5 = pd.concat(zones, ignore_index=True)
display(b5[['nom_region', 'nom_commune', 'prix_m2_median',
            'ecart_prix_region_pct', 'score_equipements', 'revenu_median']])


## Synthèse

- B2 : classement régional calculé sur les ventes, les POI et la population.
- B4 : évolution des prix par commune et type de bien.
- B5 : repérage descriptif des communes à prix bas, bien équipées et à revenu médian suffisant.

Les seuils et les limites d’interprétation sont détaillés dans `docs/gold/specification.md`.
